In [1]:
# Cell 1 — Load clean feature set
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (f1_score, balanced_accuracy_score, 
                             recall_score, precision_score)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../data/features_clean.csv")

feature_cols = ['gdp_growth_lag1', 'remittance_pct_gdp_lag1',
                'india_gdp_growth_lag1', 'current_account_lag1',
                'gdp_growth_roll3_std', 'nepal_india_inflation_spread_lag1',
                'forex_rapid_decline_lag1']

X = df[feature_cols].values
y = df['crisis'].values
years = df['year'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Crisis years: {years[y==1].tolist()}")
print(f"Crisis rate: {y.mean()*100:.1f}%")

X shape: (26, 7)
y shape: (26,)
Crisis years: [1998, 2008, 2009, 2010, 2011, 2016, 2021]
Crisis rate: 26.9%


In [2]:
# Cell 2 — LOOCV evaluation function
# For each fold: train on 25 years, test on 1 year
# Minimum training size enforced: 12 years
# Returns predictions and probabilities for all years

def run_loocv(model, X, y, years, scale=True):
    loo = LeaveOneOut()
    y_true = []
    y_pred = []
    y_prob = []
    test_years = []

    for train_idx, test_idx in loo.split(X):
        # Enforce minimum 12 year training window
        if len(train_idx) < 12:
            continue

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        if scale:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        # Get probability if model supports it
        if hasattr(model, 'predict_proba'):
            prob = model.predict_proba(X_test)[0][1]
        else:
            prob = float(pred[0])

        y_true.append(y_test[0])
        y_pred.append(pred[0])
        y_prob.append(prob)
        test_years.append(years[test_idx[0]])

    return np.array(y_true), np.array(y_pred), np.array(y_prob), np.array(test_years)

def evaluate(y_true, y_pred, model_name):
    f1  = f1_score(y_true, y_pred, zero_division=0)
    bal = balanced_accuracy_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred, zero_division=0)
    pre = precision_score(y_true, y_pred, zero_division=0)
    print(f"{model_name:<25} F1={f1:.3f}  BalAcc={bal:.3f}  Recall={rec:.3f}  Precision={pre:.3f}")
    return {'model': model_name, 'f1': f1, 'bal_acc': bal, 'recall': rec, 'precision': pre}

print("LOOCV function ready")

LOOCV function ready


In [3]:
# Cell 3 — Train and evaluate all models
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000),
    "Linear SVM":          SVC(kernel='linear', class_weight='balanced', probability=True),
    "Random Forest":       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
}

results = []
predictions = {}

for name, model in models.items():
    y_true, y_pred, y_prob, test_years = run_loocv(model, X, y, years)
    metrics = evaluate(y_true, y_pred, name)
    results.append(metrics)
    predictions[name] = {
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'years': test_years
    }

print()
print("Done.")

Logistic Regression       F1=0.429  BalAcc=0.609  Recall=0.429  Precision=0.429
Linear SVM                F1=0.400  BalAcc=0.583  Recall=0.429  Precision=0.375
Random Forest             F1=0.000  BalAcc=0.500  Recall=0.000  Precision=0.000

Done.


In [4]:
# Cell 4 — Install and train XGBoost
# XGBoost handles small datasets and class imbalance better
# scale_pos_weight compensates for imbalance (19 non-crisis / 7 crisis)

try:
    import xgboost as xgb
    print(f"XGBoost version: {xgb.__version__}")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "xgboost", "--break-system-packages"])
    import xgboost as xgb
    print(f"XGBoost version: {xgb.__version__}")

scale_pos_weight = (len(y) - y.sum()) / y.sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

y_true, y_pred, y_prob, test_years = run_loocv(xgb_model, X, y, years, scale=False)
xgb_metrics = evaluate(y_true, y_pred, "XGBoost")
results.append(xgb_metrics)
predictions["XGBoost"] = {
    'y_true': y_true,
    'y_pred': y_pred,
    'y_prob': y_prob,
    'years': test_years
}

XGBoost version: 1.7.6
scale_pos_weight: 2.71
XGBoost                   F1=0.167  BalAcc=0.466  Recall=0.143  Precision=0.200


In [5]:
# Cell 5 — Diagnosis: what is each model predicting?
print("Year-by-year predictions:\n")
print(f"{'Year':<6} {'Actual':<8}", end="")
for name in predictions:
    short = name.split()[0][:8]
    print(f"{short:<10}", end="")
print()

for i, year in enumerate(predictions['Logistic Regression']['years']):
    actual = predictions['Logistic Regression']['y_true'][i]
    print(f"{year:<6} {'CRISIS' if actual else 'stable':<8}", end="")
    for name in predictions:
        pred = predictions[name]['y_pred'][i]
        print(f"{'CRISIS' if pred else 'stable':<10}", end="")
    print()

print()
print("Total crisis predictions:")
for name in predictions:
    total = predictions[name]['y_pred'].sum()
    print(f"  {name}: {total} crisis predictions out of 26")

Year-by-year predictions:

Year   Actual  Logistic  Linear    Random    XGBoost   
1998   CRISIS  stable    stable    stable    stable    
1999   stable  CRISIS    stable    stable    CRISIS    
2000   stable  stable    stable    stable    stable    
2001   stable  stable    stable    stable    CRISIS    
2002   stable  stable    stable    stable    stable    
2004   stable  stable    stable    stable    stable    
2005   stable  stable    stable    stable    stable    
2006   stable  stable    stable    stable    stable    
2007   stable  stable    stable    stable    stable    
2008   CRISIS  CRISIS    CRISIS    stable    CRISIS    
2009   CRISIS  stable    stable    stable    stable    
2010   CRISIS  stable    stable    stable    stable    
2011   CRISIS  CRISIS    CRISIS    stable    stable    
2012   stable  CRISIS    CRISIS    stable    stable    
2013   stable  CRISIS    CRISIS    stable    stable    
2014   stable  CRISIS    CRISIS    stable    CRISIS    
2015   stable  stable

In [6]:
# Cell 6 — Feature distributions by class
# Check if features actually separate crisis from non-crisis

feature_cols = ['gdp_growth_lag1', 'remittance_pct_gdp_lag1',
                'india_gdp_growth_lag1', 'current_account_lag1',
                'gdp_growth_roll3_std', 'nepal_india_inflation_spread_lag1',
                'forex_rapid_decline_lag1']

df_check = pd.read_csv("../data/features_clean.csv")

print(f"{'Feature':<40} {'Crisis mean':>12} {'Stable mean':>12} {'Diff':>8}")
print("-" * 76)
for col in feature_cols:
    crisis_mean = df_check[df_check['crisis']==1][col].mean()
    stable_mean = df_check[df_check['crisis']==0][col].mean()
    diff = abs(crisis_mean - stable_mean)
    print(f"{col:<40} {crisis_mean:>12.3f} {stable_mean:>12.3f} {diff:>8.3f}")

Feature                                   Crisis mean  Stable mean     Diff
----------------------------------------------------------------------------
gdp_growth_lag1                                 3.549        4.614    1.065
remittance_pct_gdp_lag1                        19.467       17.163    2.304
india_gdp_growth_lag1                           4.650        6.897    2.246
current_account_lag1                            1.024       -1.194    2.218
gdp_growth_roll3_std                            1.691        2.029    0.338
nepal_india_inflation_spread_lag1              -0.965        0.712    1.677
forex_rapid_decline_lag1                        0.000        0.158    0.158


In [7]:
# Cell 7 — Option 1: Add inflation_lag1 back
# Dropped earlier due to collinearity with spread
# But inflation > 8% triggered most crisis labels directly
# Testing if the direct signal outweighs the collinearity cost

df_check = pd.read_csv("../data/features_clean.csv")
master = pd.read_csv("../data/master_labeled.csv")

# Rebuild inflation lag from master
df_check['inflation_lag1'] = master['inflation_yoy'].shift(1)
df_check = df_check.dropna().reset_index(drop=True)

feature_cols_v1 = ['gdp_growth_lag1', 'remittance_pct_gdp_lag1',
                   'india_gdp_growth_lag1', 'current_account_lag1',
                   'gdp_growth_roll3_std', 'nepal_india_inflation_spread_lag1',
                   'forex_rapid_decline_lag1', 'inflation_lag1']

X_v1 = df_check[feature_cols_v1].values
y_v1 = df_check['crisis'].values
years_v1 = df_check['year'].values

print("Option 1 — with inflation_lag1:")
print(f"Shape: {X_v1.shape}")

results_v1 = []
for name, model in models.items():
    y_true, y_pred, y_prob, test_years = run_loocv(model, X_v1, y_v1, years_v1)
    metrics = evaluate(y_true, y_pred, name)
    results_v1.append(metrics)

# XGBoost
scale_pos_weight = (len(y_v1) - y_v1.sum()) / y_v1.sum()
xgb_model2 = xgb.XGBClassifier(
    n_estimators=100, max_depth=2, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric='logloss', verbosity=0
)
y_true, y_pred, y_prob, test_years = run_loocv(xgb_model2, X_v1, y_v1, years_v1, scale=False)
xgb_v1 = evaluate(y_true, y_pred, "XGBoost")
results_v1.append(xgb_v1)

Option 1 — with inflation_lag1:
Shape: (25, 8)
Logistic Regression       F1=0.333  BalAcc=0.561  Recall=0.333  Precision=0.333
Linear SVM                F1=0.267  BalAcc=0.482  Recall=0.333  Precision=0.222
Random Forest             F1=0.000  BalAcc=0.500  Recall=0.000  Precision=0.000
XGBoost                   F1=0.167  BalAcc=0.452  Recall=0.167  Precision=0.167


In [8]:
# Cell 8 — Option 2: Add binary trigger features t2_forex and t3_inflation
# These are direct threshold crossings from Phase 1
# t2_forex = 1 if prior year forex < 7 months
# t3_inflation = 1 if prior year inflation > 8%
# Both derived from t-1 data, no leakage

df_check2 = pd.read_csv("../data/features_clean.csv")
master = pd.read_csv("../data/master_labeled.csv")

# Rebuild lagged binary triggers from master
df_check2['t2_forex_lag1']     = master['t2_forex'].shift(1)
df_check2['t3_inflation_lag1'] = master['t3_inflation'].shift(1)
df_check2 = df_check2.dropna().reset_index(drop=True)

feature_cols_v2 = ['gdp_growth_lag1', 'remittance_pct_gdp_lag1',
                   'india_gdp_growth_lag1', 'current_account_lag1',
                   'gdp_growth_roll3_std', 'nepal_india_inflation_spread_lag1',
                   'forex_rapid_decline_lag1',
                   't2_forex_lag1', 't3_inflation_lag1']

X_v2 = df_check2[feature_cols_v2].values
y_v2 = df_check2['crisis'].values
years_v2 = df_check2['year'].values

print("Option 2 — with binary triggers t2_forex_lag1 and t3_inflation_lag1:")
print(f"Shape: {X_v2.shape}")

results_v2 = []
for name, model in models.items():
    y_true, y_pred, y_prob, test_years = run_loocv(model, X_v2, y_v2, years_v2)
    metrics = evaluate(y_true, y_pred, name)
    results_v2.append(metrics)

scale_pos_weight = (len(y_v2) - y_v2.sum()) / y_v2.sum()
xgb_model3 = xgb.XGBClassifier(
    n_estimators=100, max_depth=2, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric='logloss', verbosity=0
)
y_true, y_pred, y_prob, test_years = run_loocv(xgb_model3, X_v2, y_v2, years_v2, scale=False)
xgb_v2 = evaluate(y_true, y_pred, "XGBoost")
results_v2.append(xgb_v2)

Option 2 — with binary triggers t2_forex_lag1 and t3_inflation_lag1:
Shape: (25, 9)
Logistic Regression       F1=0.462  BalAcc=0.645  Recall=0.500  Precision=0.429
Linear SVM                F1=0.462  BalAcc=0.645  Recall=0.500  Precision=0.429
Random Forest             F1=0.000  BalAcc=0.500  Recall=0.000  Precision=0.000
XGBoost                   F1=0.200  BalAcc=0.504  Recall=0.167  Precision=0.250


In [9]:
# Cell 9 — Reduced feature set: 4 features only
# Selected by crisis correlation strength and economic logic
# nepal_india_inflation_spread_lag1: -0.354 (strongest)
# india_gdp_growth_lag1:             -0.322 (strong, regional transmission)
# current_account_lag1:               0.209 (external balance)
# gdp_growth_lag1:                   -0.217 (direct growth signal)
# Dropped: remittance, roll3_std, forex_flag — weakest signals

df_v4 = pd.read_csv("../data/features_clean.csv")

feature_cols_v4 = [
    'nepal_india_inflation_spread_lag1',
    'india_gdp_growth_lag1',
    'current_account_lag1',
    'gdp_growth_lag1'
]

X_v4 = df_v4[feature_cols_v4].values
y_v4 = df_v4['crisis'].values
years_v4 = df_v4['year'].values

print("Reduced feature set — 4 features:")
print(f"Shape: {X_v4.shape}")
print()

results_v4 = []
for name, model in models.items():
    y_true, y_pred, y_prob, test_years = run_loocv(model, X_v4, y_v4, years_v4)
    metrics = evaluate(y_true, y_pred, name)
    results_v4.append(metrics)

scale_pos_weight = (len(y_v4) - y_v4.sum()) / y_v4.sum()
xgb_model5 = xgb.XGBClassifier(
    n_estimators=50, max_depth=2, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric='logloss', verbosity=0
)
y_true, y_pred, y_prob, test_years = run_loocv(xgb_model5, X_v4, y_v4, years_v4, scale=False)
xgb_v4 = evaluate(y_true, y_pred, "XGBoost")
results_v4.append(xgb_v4)

Reduced feature set — 4 features:
Shape: (26, 4)

Logistic Regression       F1=0.353  BalAcc=0.530  Recall=0.429  Precision=0.300
Linear SVM                F1=0.333  BalAcc=0.504  Recall=0.429  Precision=0.273
Random Forest             F1=0.222  BalAcc=0.545  Recall=0.143  Precision=0.500
XGBoost                   F1=0.308  BalAcc=0.538  Recall=0.286  Precision=0.333


In [10]:
# Cell 10 — Trigger-based feature set
# Using lagged binary triggers as primary features
# These are the most direct crisis signals we have
# t1=GDP<2%, t2=forex<7mo, t3=inflation>8%
# Plus two continuous features with strongest signal

df_v5 = pd.read_csv("../data/features_clean.csv")
master = pd.read_csv("../data/master_labeled.csv")

# Lagged triggers from master
df_v5['t1_gdp_lag1']          = master['t1_gdp'].shift(1)
df_v5['t2_forex_lag1']        = master['t2_forex'].shift(1)
df_v5['t3_inflation_lag1']    = master['t3_inflation'].shift(1)
df_v5 = df_v5.dropna().reset_index(drop=True)

feature_cols_v5 = [
    't1_gdp_lag1',
    't2_forex_lag1',
    't3_inflation_lag1',
    'nepal_india_inflation_spread_lag1',
    'india_gdp_growth_lag1'
]

X_v5 = df_v5[feature_cols_v5].values
y_v5 = df_v5['crisis'].values
years_v5 = df_v5['year'].values

print("Trigger-based feature set — 5 features:")
print(f"Shape: {X_v5.shape}")
print()

results_v5 = []
for name, model in models.items():
    y_true, y_pred, y_prob, test_years = run_loocv(model, X_v5, y_v5, years_v5)
    metrics = evaluate(y_true, y_pred, name)
    results_v5.append(metrics)

scale_pos_weight = (len(y_v5) - y_v5.sum()) / y_v5.sum()
xgb_model6 = xgb.XGBClassifier(
    n_estimators=50, max_depth=2, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric='logloss', verbosity=0
)
y_true, y_pred, y_prob, test_years = run_loocv(xgb_model6, X_v5, y_v5, years_v5, scale=False)
xgb_v5 = evaluate(y_true, y_pred, "XGBoost")
results_v5.append(xgb_v5)

Trigger-based feature set — 5 features:
Shape: (25, 5)

Logistic Regression       F1=0.235  BalAcc=0.430  Recall=0.333  Precision=0.182
Linear SVM                F1=0.267  BalAcc=0.482  Recall=0.333  Precision=0.222
Random Forest             F1=0.000  BalAcc=0.474  Recall=0.000  Precision=0.000
XGBoost                   F1=0.000  BalAcc=0.368  Recall=0.000  Precision=0.000


In [11]:
# Cell 11 — Path A: Threshold tuning
# Default threshold is 0.5 — lower it to boost recall
# Test thresholds from 0.2 to 0.5 in steps of 0.05
# Use Option 2 feature set (best so far: F1=0.462)

df_a = pd.read_csv("../data/features_clean.csv")
master = pd.read_csv("../data/master_labeled.csv")

df_a['t2_forex_lag1']     = master['t2_forex'].shift(1)
df_a['t3_inflation_lag1'] = master['t3_inflation'].shift(1)
df_a = df_a.dropna().reset_index(drop=True)

feature_cols_a = ['gdp_growth_lag1', 'remittance_pct_gdp_lag1',
                  'india_gdp_growth_lag1', 'current_account_lag1',
                  'gdp_growth_roll3_std', 'nepal_india_inflation_spread_lag1',
                  'forex_rapid_decline_lag1',
                  't2_forex_lag1', 't3_inflation_lag1']

X_a = df_a[feature_cols_a].values
y_a = df_a['crisis'].values
years_a = df_a['year'].values

# Get probabilities from Logistic Regression under LOOCV
loo = LeaveOneOut()
y_true_all = []
y_prob_all = []
test_years_all = []

for train_idx, test_idx in loo.split(X_a):
    if len(train_idx) < 12:
        continue
    X_train, X_test = X_a[train_idx], X_a[test_idx]
    y_train = y_a[train_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = LogisticRegression(class_weight='balanced', max_iter=1000)
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[0][1]

    y_true_all.append(y_a[test_idx[0]])
    y_prob_all.append(prob)
    test_years_all.append(years_a[test_idx[0]])

y_true_all = np.array(y_true_all)
y_prob_all = np.array(y_prob_all)

print(f"{'Threshold':<12} {'F1':>6} {'BalAcc':>8} {'Recall':>8} {'Precision':>10} {'Crisis predicted':>16}")
print("-" * 65)
best_f1 = 0
best_thresh = 0.5
for thresh in np.arange(0.20, 0.55, 0.05):
    y_pred_t = (y_prob_all >= thresh).astype(int)
    f1  = f1_score(y_true_all, y_pred_t, zero_division=0)
    bal = balanced_accuracy_score(y_true_all, y_pred_t)
    rec = recall_score(y_true_all, y_pred_t, zero_division=0)
    pre = precision_score(y_true_all, y_pred_t, zero_division=0)
    n_pred = y_pred_t.sum()
    print(f"{thresh:<12.2f} {f1:>6.3f} {bal:>8.3f} {rec:>8.3f} {pre:>10.3f} {n_pred:>16}")
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print()
print(f"Best threshold: {best_thresh:.2f} with F1={best_f1:.3f}")

Threshold        F1   BalAcc   Recall  Precision Crisis predicted
-----------------------------------------------------------------
0.20          0.300    0.461    0.500      0.214               14
0.25          0.316    0.487    0.500      0.231               13
0.30          0.316    0.487    0.500      0.231               13
0.35          0.353    0.539    0.500      0.273               11
0.40          0.429    0.618    0.500      0.375                8
0.45          0.429    0.618    0.500      0.375                8
0.50          0.462    0.645    0.500      0.429                7

Best threshold: 0.50 with F1=0.462


In [12]:
# Cell 12 — Path B: Recover 1996 crisis year
# Rolling features forced us to start from 1998, losing 1996
# Drop rolling features, use only lag features
# This recovers 1996 giving us 8 crisis years instead of 7

master = pd.read_csv("../data/master_labeled.csv")

df_b = pd.DataFrame()
df_b['year'] = master['year']

# Lag features only — no rolling
df_b['gdp_growth_lag1']                   = master['gdp_growth'].shift(1)
df_b['india_gdp_growth_lag1']             = master['india_gdp_growth'].shift(1)
df_b['current_account_lag1']              = master['current_account_pct_gdp'].shift(1)
df_b['remittance_pct_gdp_lag1']           = master['remittance_pct_gdp'].shift(1)
df_b['nepal_india_inflation_spread_lag1'] = (
    master['inflation_yoy'].shift(1) - master['india_inflation'].shift(1)
)
df_b['t2_forex_lag1']                     = master['t2_forex'].shift(1)
df_b['t3_inflation_lag1']                 = master['t3_inflation'].shift(1)
df_b['crisis']                            = master['crisis']

df_b = df_b.dropna().reset_index(drop=True)

feature_cols_b = ['gdp_growth_lag1', 'india_gdp_growth_lag1',
                  'current_account_lag1', 'remittance_pct_gdp_lag1',
                  'nepal_india_inflation_spread_lag1',
                  't2_forex_lag1', 't3_inflation_lag1']

X_b = df_b[feature_cols_b].values
y_b = df_b['crisis'].values
years_b = df_b['year'].values

print(f"Shape: {X_b.shape}")
print(f"Years: {years_b.min()} - {years_b.max()}")
print(f"Crisis years: {years_b[y_b==1].tolist()}")
print(f"Crisis rate: {y_b.mean()*100:.1f}%")
print()

results_b = []
for name, model in models.items():
    y_true, y_pred, y_prob, test_years = run_loocv(model, X_b, y_b, years_b)
    metrics = evaluate(y_true, y_pred, name)
    results_b.append(metrics)

scale_pos_weight = (len(y_b) - y_b.sum()) / y_b.sum()
xgb_b = xgb.XGBClassifier(
    n_estimators=50, max_depth=2, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric='logloss', verbosity=0
)
y_true, y_pred, y_prob, test_years = run_loocv(xgb_b, X_b, y_b, years_b, scale=False)
evaluate(y_true, y_pred, "XGBoost")

Shape: (29, 7)
Years: 1996 - 2024
Crisis years: [1996, 1998, 2008, 2009, 2010, 2011, 2016, 2021]
Crisis rate: 27.6%

Logistic Regression       F1=0.400  BalAcc=0.560  Recall=0.500  Precision=0.333
Linear SVM                F1=0.300  BalAcc=0.473  Recall=0.375  Precision=0.250
Random Forest             F1=0.200  BalAcc=0.539  Recall=0.125  Precision=0.500
XGBoost                   F1=0.533  BalAcc=0.679  Recall=0.500  Precision=0.571


{'model': 'XGBoost',
 'f1': 0.5333333333333333,
 'bal_acc': 0.6785714285714286,
 'recall': 0.5,
 'precision': 0.5714285714285714}

In [13]:
# Cell 13 — Path C: Ensemble voting on Path B dataset
# If any 2 of 3 models predict crisis, call it crisis
# Using Path B dataset (29 rows, 8 crisis years)

loo = LeaveOneOut()

# Collect predictions from all models
ensemble_preds = {
    'Logistic Regression': [],
    'Linear SVM': [],
    'XGBoost': []
}
y_true_ensemble = []
years_ensemble = []

xgb_ens = xgb.XGBClassifier(
    n_estimators=50, max_depth=2, learning_rate=0.05,
    scale_pos_weight=(len(y_b)-y_b.sum())/y_b.sum(),
    random_state=42, eval_metric='logloss', verbosity=0
)

model_map = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000),
    'Linear SVM': SVC(kernel='linear', class_weight='balanced', probability=True),
    'XGBoost': xgb_ens
}

for train_idx, test_idx in loo.split(X_b):
    if len(train_idx) < 12:
        continue

    y_true_ensemble.append(y_b[test_idx[0]])
    years_ensemble.append(years_b[test_idx[0]])

    for name, model in model_map.items():
        X_train, X_test = X_b[train_idx], X_b[test_idx]
        y_train = y_b[train_idx]

        if name != 'XGBoost':
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

        model.fit(X_train, y_train)
        pred = model.predict(X_test)[0]
        ensemble_preds[name].append(pred)

y_true_ens = np.array(y_true_ensemble)
years_ens = np.array(years_ensemble)

# Majority vote: crisis if 2 or more models predict crisis
votes = (np.array(ensemble_preds['Logistic Regression']) +
         np.array(ensemble_preds['Linear SVM']) +
         np.array(ensemble_preds['XGBoost']))

y_pred_majority = (votes >= 2).astype(int)
y_pred_any      = (votes >= 1).astype(int)

print("Ensemble — majority vote (2 of 3):")
evaluate(y_true_ens, y_pred_majority, "Majority vote")

print()
print("Ensemble — any vote (1 of 3):")
evaluate(y_true_ens, y_pred_any, "Any vote")

print()
print(f"{'Year':<6} {'Actual':<8} {'LR':<8} {'SVM':<8} {'XGB':<8} {'Majority':<10} {'Any':<6}")
print("-" * 55)
for i, year in enumerate(years_ens):
    actual = y_true_ens[i]
    lr  = ensemble_preds['Logistic Regression'][i]
    svm = ensemble_preds['Linear SVM'][i]
    xgb_p = ensemble_preds['XGBoost'][i]
    maj = y_pred_majority[i]
    any_v = y_pred_any[i]
    print(f"{year:<6} {'C' if actual else 's':<8} {'C' if lr else 's':<8} {'C' if svm else 's':<8} {'C' if xgb_p else 's':<8} {'C' if maj else 's':<10} {'C' if any_v else 's':<6}")

Ensemble — majority vote (2 of 3):
Majority vote             F1=0.316  BalAcc=0.497  Recall=0.375  Precision=0.273

Ensemble — any vote (1 of 3):
Any vote                  F1=0.522  BalAcc=0.661  Recall=0.750  Precision=0.400

Year   Actual   LR       SVM      XGB      Majority   Any   
-------------------------------------------------------
1996   C        s        s        C        s          C     
1997   s        s        s        s        s          s     
1998   C        s        s        C        s          C     
1999   s        C        C        C        C          C     
2000   s        s        C        s        s          C     
2001   s        C        C        C        C          C     
2002   s        s        s        s        s          s     
2003   s        C        C        s        C          C     
2004   s        s        s        s        s          s     
2005   s        s        s        s        s          s     
2006   s        C        C        s        C  